### 數據處理

In [1]:
import os
import glob
import json
import pandas as pd
import numpy as np

# ==================== 1. 設定實驗資料夾對照表 ====================
EXPERIMENT_CONFIGS = [
    # --- 1. 加入權威，無 Memory/Reflection ---
    {"folder": "social-simulate-authority-topology-v2-deepseek_r1_8B-result",
        "model": "DeepSeek-R1:8B", "condition": "Authority (No Memory)"},
    {"folder": "social-simulate-authority-topology-v2-llama3_8B-result",
        "model": "LLaMA3:8B", "condition": "Authority (No Memory)"},
    {"folder": "social-simulate-authority-topology-v2-qwen3_8B-result",
        "model": "Qwen3:8B", "condition": "Authority (No Memory)"},

    # --- 2. 加入權威，有 Memory/Reflection ---
    {"folder": "social-simulate-authority-topology-v3-agent-deepseek_r1_8B-result",
        "model": "DeepSeek-R1:8B", "condition": "Authority + Memory/Reflection"},
    {"folder": "social-simulate-authority-topology-v3-agent-llama3_8B-result",
        "model": "LLaMA3:8B", "condition": "Authority + Memory/Reflection"},
    {"folder": "social-simulate-authority-topology-v3-agent-qwen3_8B-result",
        "model": "Qwen3:8B", "condition": "Authority + Memory/Reflection"},

    # --- 3. 控制組（無權威），有 Memory/Reflection ---
    {"folder": "social-simulate-control-v3-agent-deepseek_r1_8B-result",
        "model": "DeepSeek-R1:8B", "condition": "Control (No Authority + Memory)"},
    {"folder": "social-simulate-control-v3-agent-llama3_8B-result",
        "model": "LLaMA3:8B", "condition": "Control (No Authority + Memory)"},
    {"folder": "social-simulate-control-v3-agent-qwen3_8B-result",
        "model": "Qwen3:8B", "condition": "Control (No Authority + Memory)"},
]


def calculate_authority_success_rate(agent_details_json_str):
    """
    解析 agent_details_json，計算單一 Run 的權威引導成功率
    """
    try:
        details = json.loads(agent_details_json_str)
    except Exception:
        return np.nan

    auth_init_stance = None
    auth_name = None

    # 1. 找出權威是誰以及其初始立場
    for name, info in details.items():
        if info.get("is_authority", False):
            auth_name = name
            auth_init_stance = info.get("initial_stance", "")
            break

    # 若無權威（控制組），回傳 N/A
    if not auth_name or not auth_init_stance:
        return np.nan

    non_auth_count = 0
    swayed_towards_auth_count = 0

    # 2. 檢測非權威 Agent 的立場位移方向是否與權威一致
    for name, info in details.items():
        if name == auth_name:
            continue

        non_auth_count += 1
        init_s = info.get("initial_score", 5.0)
        final_s = info.get("final_score", 5.0)

        # 權威立場為 Support (1-10分中 > 5 分)
        if auth_init_stance == "Support" and final_s > init_s:
            swayed_towards_auth_count += 1
        # 權威立場為 Oppose (1-10分中 < 5 分)
        elif auth_init_stance == "Oppose" and final_s < init_s:
            swayed_towards_auth_count += 1

    if non_auth_count == 0:
        return 0.0

    return swayed_towards_auth_count / non_auth_count


def process_single_folder(folder_path):
    """
    讀取資料夾內的 *_metrics.csv 並計算該組別的平均/最大量化指標
    """
    csv_pattern = os.path.join(folder_path, "*_metrics.csv")
    csv_files = glob.glob(csv_pattern)

    if not csv_files:
        print(f"⚠️ 警告：在路徑 '{folder_path}' 中找不到 *_metrics.csv 檔案。")
        return None

    # 預設讀取最新的 CSV 檔案
    target_csv = sorted(csv_files)[-1]
    df = pd.read_csv(target_csv)

    # 計算單組內每筆 Run 的權威引導成功率
    if "agent_details_json" in df.columns:
        df["auth_success_rate"] = df["agent_details_json"].apply(
            calculate_authority_success_rate)
    else:
        df["auth_success_rate"] = np.nan

    # 7 大核心指標統計
    metrics_summary = {
        "runs_count": len(df),
        "mean_shift": round(df["mean_shift"].mean(), 3) if "mean_shift" in df else np.nan,
        "polarization_index": round(df["polarization_index"].mean(), 3) if "polarization_index" in df else np.nan,
        "fragmentation_index": round(df["fragmentation_index"].mean(), 3) if "fragmentation_index" in df else np.nan,
        "conformity_rate": round(df["conformity_rate"].mean(), 3) if "conformity_rate" in df else np.nan,
        "total_authority_in_degree": int(df["authority_in_degree"].sum()) if "authority_in_degree" in df else 0,
        "mean_authority_in_degree": round(df["authority_in_degree"].mean(), 2) if "authority_in_degree" in df else 0.0,
        "max_cascade_depth": int(df["cascade_depth"].max()) if "cascade_depth" in df else 0,
        "mean_cascade_depth": round(df["cascade_depth"].mean(), 2) if "cascade_depth" in df else 0.0,
        "authority_success_rate": round(df["auth_success_rate"].dropna().mean() * 100, 2) if not df["auth_success_rate"].dropna().empty else "N/A (控制組)"
    }

    return metrics_summary


def main():
    print("=" * 80)
    print(" 🚀 開始執行社會模擬實驗數據整合分析腳本")
    print(" 整合對比：DeepSeek-R1 / LLaMA3 / Qwen3 在三種權威與記憶條件下的 7 大社會學指標")
    print("=" * 80 + "\n")

    summary_list = []

    for config in EXPERIMENT_CONFIGS:
        folder = config["folder"]
        model = config["model"]
        condition = config["condition"]

        print(f"🔍 正在處理：[{model}] - [{condition}]...")
        res = process_single_folder(folder)

        if res:
            res["Model"] = model
            res["Condition"] = condition
            res["Folder"] = folder
            summary_list.append(res)

    if not summary_list:
        print("❌ 未發現任何可供分析的數據，請檢查資料夾名稱與路徑！")
        return

    # 整理成 DataFrame
    summary_df = pd.DataFrame(summary_list)

    # 調整顯示順序與欄位名稱
    columns_order = [
        "Model", "Condition", "runs_count",
        "mean_shift", "polarization_index", "fragmentation_index", "conformity_rate",
        "mean_authority_in_degree", "total_authority_in_degree",
        "mean_cascade_depth", "max_cascade_depth",
        "authority_success_rate"
    ]
    summary_df = summary_df[columns_order]

    rename_map = {
        "Model": "模型名稱",
        "Condition": "實驗條件",
        "runs_count": "總重複次數",
        "mean_shift": "平均立場位移",
        "polarization_index": "群體極化指數",
        "fragmentation_index": "群體分裂指數",
        "conformity_rate": "從眾率",
        "mean_authority_in_degree": "權威次均被提及數",
        "total_authority_in_degree": "權威被提及總次數",
        "mean_cascade_depth": "級聯擴散平均深度",
        "max_cascade_depth": "級聯擴散最大深度",
        "authority_success_rate": "權威引導成功率(%)"
    }

    display_df = summary_df.rename(columns=rename_map)

    print("\n" + "=" * 100)
    print("📊 實驗數據分析總結對比表 (Summary Matrix)")
    print("=" * 100)
    print(display_df.to_string(index=False))
    print("=" * 100)

    # 匯出 CSV 檔供論文繪圖與撰寫表格使用
    output_filename = "simulation_analysis_summary_matrix.csv"
    display_df.to_csv(output_filename, index=False, encoding="utf-8-sig")
    print(f"\n✅ 分析報告已成功匯出至：{os.path.abspath(output_filename)}")

In [2]:
main()

 🚀 開始執行社會模擬實驗數據整合分析腳本
 整合對比：DeepSeek-R1 / LLaMA3 / Qwen3 在三種權威與記憶條件下的 7 大社會學指標

🔍 正在處理：[DeepSeek-R1:8B] - [Authority (No Memory)]...
🔍 正在處理：[LLaMA3:8B] - [Authority (No Memory)]...
🔍 正在處理：[Qwen3:8B] - [Authority (No Memory)]...
🔍 正在處理：[DeepSeek-R1:8B] - [Authority + Memory/Reflection]...
🔍 正在處理：[LLaMA3:8B] - [Authority + Memory/Reflection]...
🔍 正在處理：[Qwen3:8B] - [Authority + Memory/Reflection]...
🔍 正在處理：[DeepSeek-R1:8B] - [Control (No Authority + Memory)]...
🔍 正在處理：[LLaMA3:8B] - [Control (No Authority + Memory)]...
🔍 正在處理：[Qwen3:8B] - [Control (No Authority + Memory)]...

📊 實驗數據分析總結對比表 (Summary Matrix)
          模型名稱                            實驗條件  總重複次數  平均立場位移  群體極化指數  群體分裂指數   從眾率  權威次均被提及數  權威被提及總次數  級聯擴散平均深度  級聯擴散最大深度 權威引導成功率(%)
DeepSeek-R1:8B           Authority (No Memory)     25  -0.660  -0.140   0.652 0.432     30.84       771      2.72         4       54.4
     LLaMA3:8B           Authority (No Memory)     25  -1.540  -0.114   0.472 0.406     56.64      1416      2.40       